In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


# ============================================================
# Pfade / Einstellungen
# ============================================================

PREDICTIONS_PATH = Path(
    "//Users/flo/Desktop/Isenhagen/thz-anomaly-supervised/results/hgb/hgb_predictions.csv"
)

MEASUREMENT_ROOT = Path(
    "/Users/flo/Desktop/Isenhagen/thz-anomaly-supervised/data/measurements"
)

OUTPUT_PDF = Path(
    "/Users/flo/Desktop/Isenhagen/thz-anomaly-supervised/results/critical_hgb_cases.pdf"
)

# Welche Vorhersage soll analysiert werden?
# Möglich z.B.: "pred_raw" oder "pred_threshold"
PRED_COL = "pred_threshold"

TARGET_COL = "label"

# Kritischer Fehler:
# echte Klasse 1 oder 2, aber vorhergesagt als 3
HOHLRAUM_LABELS = [1, 2]
NO_HOHLRAUM_LABEL = 3

N_CASES = 10

# Falls mehrere Pfadspalten möglich sind
POSSIBLE_PATH_COLS = [
    "relative_path",
    "file_path",
    "path",
    "measurement_path",
]


# ============================================================
# Hilfsfunktionen
# ============================================================

def find_path_col(df):
    for col in POSSIBLE_PATH_COLS:
        if col in df.columns:
            return col
    raise ValueError(
        f"Keine Pfadspalte gefunden. Erwartet eine von: {POSSIBLE_PATH_COLS}"
    )


def resolve_measurement_path(value):
    """
    Baut aus relative_path oder absolutem Pfad einen echten Dateipfad.
    """
    p = Path(str(value))

    if p.is_absolute():
        return p

    return MEASUREMENT_ROOT / p


def load_measurement_txt(path):
    """
    Lädt eine Messdatei robust ein.

    Erwartet typischerweise:
    Spalte 0 = Zeit / x-Wert
    Spalte 1 = Signal / Amplitude

    Falls mehr Spalten vorhanden sind, wird die letzte numerische Spalte
    als Signal genommen.
    """

    if not path.exists():
        raise FileNotFoundError(f"Messdatei nicht gefunden: {path}")

    df = pd.read_csv(
        path,
        sep=r"\s+|;|,",
        engine="python",
        comment="#",
        header=None
    )

    # Nur numerische Werte behalten
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all")

    if df.shape[1] == 0:
        raise ValueError(f"Keine numerischen Daten in Datei: {path}")

    if df.shape[1] == 1:
        y = df.iloc[:, 0].to_numpy()
        x = np.arange(len(y))
    else:
        x = df.iloc[:, 0].to_numpy()
        y = df.iloc[:, -1].to_numpy()

    return x, y


def short_path(path, max_len=90):
    s = str(path)
    if len(s) <= max_len:
        return s
    return "..." + s[-max_len:]


# ============================================================
# Daten laden
# ============================================================

df = pd.read_csv(PREDICTIONS_PATH)

if TARGET_COL not in df.columns:
    raise ValueError(f"Spalte '{TARGET_COL}' nicht gefunden.")

if PRED_COL not in df.columns:
    raise ValueError(f"Spalte '{PRED_COL}' nicht gefunden.")

path_col = find_path_col(df)

print("\nGeladene Vorhersagen:")
print(f"Zeilen: {len(df)}")
print(f"Pfadspalte: {path_col}")
print(f"Vorhersagespalte: {PRED_COL}")


# ============================================================
# Kritische Fälle filtern
# ============================================================

critical_df = df[
    df[TARGET_COL].isin(HOHLRAUM_LABELS)
    & (df[PRED_COL] == NO_HOHLRAUM_LABEL)
].copy()

print("\nKritische Fälle gefunden:")
print(len(critical_df))

if len(critical_df) == 0:
    raise ValueError("Keine kritischen Fälle gefunden.")


# Falls p_3 existiert: die kritischsten Fälle zuerst,
# also die, bei denen das Modell besonders sicher 'kein Hohlraum' gesagt hat.
if "p_3" in critical_df.columns:
    critical_df = critical_df.sort_values("p_3", ascending=False)
else:
    critical_df = critical_df.sample(
        n=min(N_CASES, len(critical_df)),
        random_state=42
    )

critical_df = critical_df.head(N_CASES)

print("\nAusgewählte Fälle:")
cols_to_show = [TARGET_COL, PRED_COL]
for col in ["p_0", "p_1", "p_2", "p_3", path_col]:
    if col in critical_df.columns:
        cols_to_show.append(col)

print(critical_df[cols_to_show])


# ============================================================
# Plotten und als PDF speichern
# ============================================================

OUTPUT_PDF.parent.mkdir(parents=True, exist_ok=True)

with PdfPages(OUTPUT_PDF) as pdf:
    for i, (_, row) in enumerate(critical_df.iterrows(), start=1):
        measurement_path = resolve_measurement_path(row[path_col])

        try:
            x, y = load_measurement_txt(measurement_path)
        except Exception as e:
            print(f"\nFehler beim Laden von {measurement_path}:")
            print(e)
            continue

        fig, ax = plt.subplots(figsize=(12, 5))

        ax.plot(x, y, linewidth=1.2)

        true_label = row[TARGET_COL]
        pred_label = row[PRED_COL]

        title = (
            f"Kritischer Fall {i}/{len(critical_df)} | "
            f"true={true_label}, pred={pred_label}"
        )

        if "p_3" in row:
            title += f", p_3={row['p_3']:.3f}"

        ax.set_title(title)
        ax.set_xlabel("Zeit / x")
        ax.set_ylabel("Signal / Amplitude")
        ax.grid(True, alpha=0.3)

        info_lines = [
            f"Datei: {short_path(measurement_path)}",
        ]

        for col in ["p_0", "p_1", "p_2", "p_3"]:
            if col in row:
                info_lines.append(f"{col}: {row[col]:.4f}")

        info_text = "\n".join(info_lines)

        ax.text(
            0.01,
            0.98,
            info_text,
            transform=ax.transAxes,
            verticalalignment="top",
            fontsize=9,
            bbox=dict(boxstyle="round", alpha=0.15)
        )

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"\nPDF gespeichert unter:")
print(OUTPUT_PDF)


Geladene Vorhersagen:
Zeilen: 10639
Pfadspalte: file_path
Vorhersagespalte: pred_threshold

Kritische Fälle gefunden:
40

Ausgewählte Fälle:
      label  pred_threshold       p_0       p_1       p_2       p_3  \
6834      1               3  0.003504  0.004399  0.000304  0.991793   
5204      1               3  0.001321  0.012317  0.000126  0.986236   
5275      2               3  0.002667  0.000801  0.020238  0.976294   
9212      2               3  0.020166  0.000896  0.004646  0.974293   
3632      2               3  0.003471  0.000742  0.021749  0.974037   
3038      2               3  0.011170  0.000364  0.023631  0.964835   
5211      2               3  0.003609  0.002661  0.035432  0.958298   
5085      1               3  0.014007  0.027561  0.000256  0.958176   
3787      1               3  0.005850  0.032533  0.005529  0.956088   
746       2               3  0.020365  0.000130  0.030430  0.949076   

                                              file_path  
6834  /Users/flo/D